In [ ]:
from src import analysis, common

TASK = "steady_flow"
RUN_NAME = "replace_with_current_run_name"
MODEL_TRAINING_DATA_ROOT = common.paths.get_model_training_data_root()
TRAINING_META_ROOT = common.paths.get_training_meta_root()
TRAINING_RAW_ROOT = common.paths.get_training_raw_root()
TRAINING_PROCESSED_ROOT = common.paths.get_training_processed_root()
RUN_DIR = common.paths.resolve_run_output_dir(
    TASK,
    RUN_NAME,
    output_root=TRAINING_PROCESSED_ROOT,
)
print(
    {
        "model_training_data_root": str(MODEL_TRAINING_DATA_ROOT),
        "training_meta_root": str(TRAINING_META_ROOT),
        "training_raw_root": str(TRAINING_RAW_ROOT),
        "training_processed_root": str(TRAINING_PROCESSED_ROOT),
        "run_dir": str(RUN_DIR),
    }
)

In [ ]:
# Build or validate both saved-membership artifact roles through the public service.
frames_by_run = analysis.artifacts.service.build_artifacts(
    runs_root=RUN_DIR,
    metadata_root=TRAINING_META_ROOT,
    dataset_root=TRAINING_RAW_ROOT,
    max_cases=None,
    batch_size=1,
    device_policy="cpu",
    rebuild=False,
)
raw_roles = frames_by_run[RUN_DIR.name]
datasets_eval = {
    f"{RUN_NAME} ID": analysis.evaluation.dataframe.build_eval_df(raw_roles["eval"]),
    f"{RUN_NAME} OOD": analysis.evaluation.dataframe.build_eval_df(raw_roles["ood"]),
}
analysis.evaluation.dataframe.validate_comparison(datasets_eval, require_physics=True)

In [ ]:
panel = analysis.evaluation.panel.build_evaluation_panel(
    datasets_eval=datasets_eval,
    title=RUN_NAME,
    sections="all",
)
display(panel)